In [1]:
import numpy as np
import pandas as pd
import sklearn

print("Everything works!")

Everything works!


In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project paths
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
TABLES_DIR = PROJECT_ROOT / "tables1"
FIGURES_DIR = PROJECT_ROOT / "figures1"

SUBJECT = "YFF"

print("Project root :", PROJECT_ROOT)
print("Data dir     :", DATA_DIR)
print("Subject      :", SUBJECT)

Project root : c:\Users\shafi\number-simplex-reproduction
Data dir     : c:\Users\shafi\number-simplex-reproduction\data\raw
Subject      : YFF


In [4]:
import src.number_decoding_analysis as nda

print("number_decoding_analysis imported successfully.")

number_decoding_analysis imported successfully.


In [5]:
out = nda.analyze_neuron(
    "YFF",
    9,
    root=PROJECT_ROOT,
    plot=False,
    verbose=False,
)

presentations = out["presentations"]

print("Number of presentations:", len(presentations))
print("\nColumns:")
print(presentations.columns.tolist())

display(presentations.head(10))

Number of presentations: 109

Columns:
['trial', 'number', 'onset', 'operand', 'spike_count', 'firing_rate']


,trial,number,onset,operand,spike_count,firing_rate
0,1,5,6089.466667,1,1,1.111111
1,2,2,15882.233333,1,1,1.111111
2,3,6,23189.233333,1,9,10.000000
5,6,9,43108.400000,1,4,4.444444
6,7,5,49998.333333,1,4,4.444444
7,8,8,58206.233333,1,4,4.444444
8,9,9,64245.366667,1,4,4.444444
10,11,6,78142.033333,1,1,1.111111
11,12,1,86066.300000,1,4,4.444444
12,13,2,90971.033333,1,8,8.888889


In [6]:
results_path = TABLES_DIR / "figure1M_YFF_permutation.csv"

yff_results = pd.read_csv(results_path)

print("Shape:", yff_results.shape)
print("\nColumns:")
print(yff_results.columns.tolist())

display(yff_results.head())

Shape: (37, 16)

Columns:
['subject', 'neuron', 'region', 'FR_accuracy', 'TC_accuracy', 'best_bin_ms', 'best_gamma', 'n_presentations', 'n_splits', 'FR_null_mean', 'FR_p', 'FR_coding', 'TC_null_mean', 'TC_p', 'TC_coding', 'elapsed_seconds']


,subject,neuron,region,FR_accuracy,TC_accuracy,best_bin_ms,best_gamma,n_presentations,n_splits,FR_null_mean,FR_p,FR_coding,TC_null_mean,TC_p,TC_coding,elapsed_seconds
0,YFF,9,ent,0.100917,0.165138,180,0.2,109,7,0.106376,0.646766,False,0.105092,0.059701,False,12.667940
1,YFF,10,hpc,0.091743,0.155963,90,0.2,109,7,0.106881,0.731343,False,0.109266,0.094527,False,11.077304
2,YFF,11,hpc,0.110092,0.201835,100,0.5,109,7,0.109404,0.532338,False,0.107248,0.019900,True,10.417088
3,YFF,12,hpc,0.100917,0.174312,300,0.5,109,7,0.110459,0.641791,False,0.108165,0.049751,True,11.636312
4,YFF,13,hpc,0.146789,0.155963,100,0.8,109,7,0.104771,0.159204,False,0.113257,0.139303,False,11.468347


In [7]:
neuron_ids = yff_results["neuron"].astype(int).tolist()

print("Number of neurons:", len(neuron_ids))
print("Neuron IDs:")
print(neuron_ids)

Number of neurons: 37
Neuron IDs:
[9, 10, 11, 12, 13, 14, 15, 16, 17, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54]


In [8]:
for neuron_id in neuron_ids[:5]:

    out_n = nda.analyze_neuron(
        SUBJECT,
        neuron_id,
        root=PROJECT_ROOT,
        plot=False,
        verbose=False,
    )

    p = out_n["presentations"]

    print(
        f"Neuron {neuron_id:3d} | "
        f"presentations = {len(p):3d} | "
        f"unique trials = {p['trial'].nunique():3d} | "
        f"trial range = {p['trial'].min()}–{p['trial'].max()}"
    )
    

Neuron   9 | presentations = 109 | unique trials =  82 | trial range = 1–100
Neuron  10 | presentations = 109 | unique trials =  82 | trial range = 1–100
Neuron  11 | presentations = 109 | unique trials =  82 | trial range = 1–100
Neuron  12 | presentations = 109 | unique trials =  82 | trial range = 1–100
Neuron  13 | presentations = 109 | unique trials =  82 | trial range = 1–100


In [9]:
# Use neuron 9 as the reference presentation sequence
ref_id = neuron_ids[0]

ref_out = nda.analyze_neuron(
    SUBJECT,
    ref_id,
    root=PROJECT_ROOT,
    plot=False,
    verbose=False,
)

ref_p = ref_out["presentations"].reset_index(drop=True)

alignment_cols = ["trial", "number", "onset", "operand"]

print(f"Reference neuron: {ref_id}")
print(f"Presentations: {len(ref_p)}")
print()

for neuron_id in neuron_ids[1:6]:

    out_n = nda.analyze_neuron(
        SUBJECT,
        neuron_id,
        root=PROJECT_ROOT,
        plot=False,
        verbose=False,
    )

    p = out_n["presentations"].reset_index(drop=True)

    same_length = len(p) == len(ref_p)

    if same_length:
        same_trial   = np.array_equal(p["trial"].values, ref_p["trial"].values)
        same_number  = np.array_equal(p["number"].values, ref_p["number"].values)
        same_operand = np.array_equal(p["operand"].values, ref_p["operand"].values)
        same_onset   = np.allclose(p["onset"].values, ref_p["onset"].values)

        exact_alignment = (
            same_trial
            and same_number
            and same_operand
            and same_onset
        )
    else:
        same_trial = same_number = same_operand = same_onset = False
        exact_alignment = False

    print(
        f"Neuron {neuron_id:3d} | "
        f"trial={same_trial} | "
        f"number={same_number} | "
        f"operand={same_operand} | "
        f"onset={same_onset} | "
        f"EXACT={exact_alignment}"
    )

Reference neuron: 9
Presentations: 109

Neuron  10 | trial=True | number=True | operand=True | onset=True | EXACT=True
Neuron  11 | trial=True | number=True | operand=True | onset=True | EXACT=True
Neuron  12 | trial=True | number=True | operand=True | onset=True | EXACT=True
Neuron  13 | trial=True | number=True | operand=True | onset=True | EXACT=True
Neuron  14 | trial=True | number=True | operand=True | onset=True | EXACT=True


In [10]:
alignment_results = []

for neuron_id in neuron_ids:

    out_n = nda.analyze_neuron(
        SUBJECT,
        neuron_id,
        root=PROJECT_ROOT,
        plot=False,
        verbose=False,
    )

    p = out_n["presentations"].reset_index(drop=True)

    same_length = len(p) == len(ref_p)

    if same_length:
        same_trial = np.array_equal(
            p["trial"].values,
            ref_p["trial"].values
        )

        same_number = np.array_equal(
            p["number"].values,
            ref_p["number"].values
        )

        same_operand = np.array_equal(
            p["operand"].values,
            ref_p["operand"].values
        )

        same_onset = np.allclose(
            p["onset"].values,
            ref_p["onset"].values
        )

    else:
        same_trial = False
        same_number = False
        same_operand = False
        same_onset = False

    exact = (
        same_length
        and same_trial
        and same_number
        and same_operand
        and same_onset
    )

    alignment_results.append({
        "neuron": neuron_id,
        "n_presentations": len(p),
        "same_trial": same_trial,
        "same_number": same_number,
        "same_operand": same_operand,
        "same_onset": same_onset,
        "exact_alignment": exact,
    })


alignment_df = pd.DataFrame(alignment_results)

display(alignment_df)

print("\nSummary")
print("Total neurons :", len(alignment_df))
print("Fully aligned :", alignment_df["exact_alignment"].sum())
print("Not aligned   :", (~alignment_df["exact_alignment"]).sum())

,neuron,n_presentations,same_trial,same_number,same_operand,same_onset,exact_alignment
0,9,109,True,True,True,True,True
1,10,109,True,True,True,True,True
2,11,109,True,True,True,True,True
3,12,109,True,True,True,True,True
4,13,109,True,True,True,True,True
5,14,109,True,True,True,True,True
6,15,109,True,True,True,True,True
7,16,109,True,True,True,True,True
8,17,109,True,True,True,True,True
9,27,109,True,True,True,True,True



Summary
Total neurons : 37
Fully aligned : 37
Not aligned   : 0


In [11]:
number_counts = (
    ref_p["number"]
    .value_counts()
    .sort_index()
)

print("Presentations per number:")
print(number_counts)

print("\nTotal:", number_counts.sum())

Presentations per number:
number
1    10
2    15
3    11
4     7
5     7
6    16
7    17
8    12
9    14
Name: count, dtype: int64

Total: 109


In [12]:
operand_counts = pd.crosstab(
    ref_p["number"],
    ref_p["operand"]
)

display(operand_counts)

operand,1,2
number,,
1,4,6
2,4,11
3,5,6
4,2,5
5,6,1
6,10,6
7,9,8
8,5,7
9,10,4


In [13]:
BIN_MS = 100
WINDOW_START_MS = 50
WINDOW_END_MS = 950

bin_edges = np.arange(
    WINDOW_START_MS,
    WINDOW_END_MS + BIN_MS,
    BIN_MS
)

print("Bin edges:", bin_edges)
print("Number of bins:", len(bin_edges) - 1)

Bin edges: [ 50 150 250 350 450 550 650 750 850 950]
Number of bins: 9


In [14]:
spike_times = ref_out["spike_times"]

X_neuron9 = []

for onset in ref_p["onset"]:

    relative_spikes = spike_times - onset

    counts, _ = np.histogram(
        relative_spikes,
        bins=bin_edges
    )

    X_neuron9.append(counts)

X_neuron9 = np.asarray(X_neuron9)

print("Shape:", X_neuron9.shape)
print("\nFirst 5 presentations:")
print(X_neuron9[:5])

Shape: (109, 9)

First 5 presentations:
[[0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0]
 [0 2 1 1 0 1 1 1 2]
 [0 1 0 1 0 0 1 1 0]
 [0 0 1 0 0 0 2 1 0]]


In [15]:
binned_total = X_neuron9.sum(axis=1)

original_total = ref_p["spike_count"].to_numpy()

matches = binned_total == original_total

print("Presentations checked :", len(matches))
print("Exact matches         :", matches.sum())
print("Mismatches            :", (~matches).sum())

if not matches.all():
    bad_idx = np.where(~matches)[0]
    print("\nFirst mismatch indices:", bad_idx[:10])

Presentations checked : 109
Exact matches         : 109
Mismatches            : 0


In [16]:
population_blocks = []

for neuron_id in neuron_ids:

    out_n = nda.analyze_neuron(
        SUBJECT,
        neuron_id,
        root=PROJECT_ROOT,
        plot=False,
        verbose=False,
    )

    spike_times = out_n["spike_times"]

    neuron_matrix = []

    for onset in ref_p["onset"]:

        relative_spikes = spike_times - onset

        counts, _ = np.histogram(
            relative_spikes,
            bins=bin_edges
        )

        neuron_matrix.append(counts)

    neuron_matrix = np.asarray(neuron_matrix)

    population_blocks.append(neuron_matrix)


X = np.hstack(population_blocks)

y = ref_p["number"].astype(int).to_numpy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nPresentations :", X.shape[0])
print("Features      :", X.shape[1])
print("Neurons       :", len(neuron_ids))
print("Bins/neuron   :", len(bin_edges) - 1)
print("Expected      :", len(neuron_ids) * (len(bin_edges) - 1))

X shape: (109, 333)
y shape: (109,)

Presentations : 109
Features      : 333
Neurons       : 37
Bins/neuron   : 9
Expected      : 333


In [17]:
trial_counts = (
    ref_p.groupby("trial")
    .size()
    .value_counts()
    .sort_index()
)

print("Number of presentations contributed by each trial:")
print(trial_counts)

print("\nUnique trials:", ref_p["trial"].nunique())

multi_trials = (
    ref_p.groupby("trial")
    .filter(lambda x: len(x) > 1)
)

print("Presentations belonging to multi-presentation trials:",
      len(multi_trials))

Number of presentations contributed by each trial:
1    55
2    27
Name: count, dtype: int64

Unique trials: 82
Presentations belonging to multi-presentation trials: 54


In [18]:
example_trials = (
    ref_p.groupby("trial")
    .filter(lambda x: len(x) > 1)
    [["trial", "number", "operand", "onset"]]
    .head(12)
)

display(example_trials)

,trial,number,operand,onset
1,2,2,1,15882.233333
2,3,6,1,23189.233333
3,6,9,1,43108.400000
5,8,8,1,58206.233333
6,9,9,1,64245.366667
8,12,1,1,86066.300000
10,17,6,1,117813.466667
15,27,3,1,189348.866667
16,30,6,1,211820.433333
18,33,5,1,237978.866667


In [19]:
counts_per_trial = ref_p.groupby("trial").size()

two_presentation_trials = counts_per_trial[
    counts_per_trial == 2
].index[:6]

example_pairs = (
    ref_p[
        ref_p["trial"].isin(two_presentation_trials)
    ]
    [["trial", "number", "operand", "onset"]]
    .sort_values(["trial", "onset"])
)

display(example_pairs)

,trial,number,operand,onset
1,2,2,1,15882.233333
55,2,1,2,16632.933333
2,3,6,1,23189.233333
56,3,1,2,23939.966667
3,6,9,1,43108.400000
58,6,8,2,43859.100000
5,8,8,1,58206.233333
59,8,1,2,58956.933333
6,9,9,1,64245.366667
60,9,1,2,64996.066667


In [20]:
groups = ref_p["trial"].astype(int).to_numpy()

print("X      :", X.shape)
print("y      :", y.shape)
print("groups :", groups.shape)

print("\nUnique presentations:", len(y))
print("Unique trials       :", len(np.unique(groups)))
print("Unique classes      :", np.unique(y))

X      : (109, 333)
y      : (109,)
groups : (109,)

Unique presentations: 109
Unique trials       : 82
Unique classes      : [1 2 3 4 5 6 7 8 9]


In [21]:
from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

splits = list(cv.split(X, y, groups=groups))

for fold, (train_idx, test_idx) in enumerate(splits, start=1):

    print(
        f"Fold {fold}: "
        f"train={len(train_idx):3d}, "
        f"test={len(test_idx):3d}, "
        f"train trials={len(np.unique(groups[train_idx])):2d}, "
        f"test trials={len(np.unique(groups[test_idx])):2d}"
    )

Fold 1: train= 86, test= 23, train trials=66, test trials=16
Fold 2: train= 87, test= 22, train trials=65, test trials=17
Fold 3: train= 88, test= 21, train trials=66, test trials=16
Fold 4: train= 87, test= 22, train trials=65, test trials=17
Fold 5: train= 88, test= 21, train trials=66, test trials=16


In [22]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score

lda = LinearDiscriminantAnalysis(
    solver="lsqr",
    shrinkage="auto"
)

logistic = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=5000,
        solver="lbfgs"
    )
)

results = []

for fold, (train_idx, test_idx) in enumerate(splits, start=1):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # ----- LDA -----
    lda.fit(X_train, y_train)
    y_pred_lda = lda.predict(X_test)

    # ----- Logistic regression -----
    logistic.fit(X_train, y_train)
    y_pred_log = logistic.predict(X_test)

    results.append({
        "fold": fold,
        "lda_accuracy": accuracy_score(y_test, y_pred_lda),
        "lda_balanced_accuracy": balanced_accuracy_score(y_test, y_pred_lda),
        "logistic_accuracy": accuracy_score(y_test, y_pred_log),
        "logistic_balanced_accuracy": balanced_accuracy_score(y_test, y_pred_log),
    })

results_df = pd.DataFrame(results)

display(results_df)

print("\nMean CV results")
print("-" * 45)
print(
    f"LDA accuracy:               "
    f"{results_df['lda_accuracy'].mean():.3f}"
)
print(
    f"LDA balanced accuracy:      "
    f"{results_df['lda_balanced_accuracy'].mean():.3f}"
)
print(
    f"Logistic accuracy:          "
    f"{results_df['logistic_accuracy'].mean():.3f}"
)
print(
    f"Logistic balanced accuracy: "
    f"{results_df['logistic_balanced_accuracy'].mean():.3f}"
)

print(f"\nNominal chance: {1/9:.3f}")

c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: 

,fold,lda_accuracy,lda_balanced_accuracy,logistic_accuracy,logistic_balanced_accuracy
0,1,0.086957,0.071429,0.173913,0.166667
1,2,0.136364,0.150000,0.090909,0.087500
2,3,0.238095,0.277778,0.238095,0.277778
3,4,0.181818,0.156250,0.181818,0.177083
4,5,0.142857,0.100000,0.047619,0.035714



Mean CV results
---------------------------------------------
LDA accuracy:               0.157
LDA balanced accuracy:      0.151
Logistic accuracy:          0.146
Logistic balanced accuracy: 0.149

Nominal chance: 0.111


In [23]:
N_PERM = 100
rng = np.random.default_rng(42)

lda_null = []
logistic_null = []

for perm in range(N_PERM):

    y_perm = rng.permutation(y)

    lda_fold_acc = []
    log_fold_acc = []

    for train_idx, test_idx in splits:

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y_perm[train_idx]
        y_test = y_perm[test_idx]

        # LDA
        lda.fit(X_train, y_train)
        lda_pred = lda.predict(X_test)

        lda_fold_acc.append(
            accuracy_score(y_test, lda_pred)
        )

        # Logistic regression
        logistic.fit(X_train, y_train)
        log_pred = logistic.predict(X_test)

        log_fold_acc.append(
            accuracy_score(y_test, log_pred)
        )

    lda_null.append(np.mean(lda_fold_acc))
    logistic_null.append(np.mean(log_fold_acc))


lda_null = np.asarray(lda_null)
logistic_null = np.asarray(logistic_null)

lda_observed = results_df["lda_accuracy"].mean()
logistic_observed = results_df["logistic_accuracy"].mean()

print("LDA")
print("Observed accuracy :", round(lda_observed, 3))
print("Null mean         :", round(lda_null.mean(), 3))
print("Null std          :", round(lda_null.std(), 3))

print("\nLogistic")
print("Observed accuracy :", round(logistic_observed, 3))
print("Null mean         :", round(logistic_null.mean(), 3))
print("Null std          :", round(logistic_null.std(), 3))

c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\covariance\_shrunk_covariance.py:349: UserWarning: Only one sample available. You may want to reshape your data array
  warnings.warn(
c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\covariance\_empirical_covariance.py:102: UserWarning: Only one sample available. You may want to reshape your data array
  warnings.warn(


LDA
Observed accuracy : 0.157
Null mean         : 0.113
Null std          : 0.033

Logistic
Observed accuracy : 0.146
Null mean         : 0.11
Null std          : 0.031


In [24]:
from src.population_level_decoding import analyze_subject_population

test = analyze_subject_population(
    subject="YFF",
    neuron_ids=neuron_ids,
    root=PROJECT_ROOT,
    bin_ms=100,
)

POPULATION DECODING: YFF
Neurons       : 37
Presentations : 109
Trials        : 82
Bins/neuron   : 9
Features      : 333
X shape       : (109, 333)


c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



Decoding results
------------------------------------------------------------
LDA accuracy               : 0.157
LDA balanced accuracy      : 0.151
Logistic accuracy          : 0.146
Logistic balanced accuracy : 0.149
Nominal chance             : 0.111

Saved summary : c:\Users\shafi\number-simplex-reproduction\tables1\population_decoding\YFF_population_summary.csv
Saved folds   : c:\Users\shafi\number-simplex-reproduction\tables1\population_decoding\YFF_population_folds.csv


c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\shafi\number-simplex-reproduction\.venv312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [25]:
from sklearn.model_selection import StratifiedGroupKFold
import sklearn

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

print("sklearn:", sklearn.__version__)
print("X shape:", X.shape)

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X, y, groups=groups), start=1
):
    test_trials = sorted(set(groups[test_idx]))

    print(f"\nFold {fold}")
    print("n_test presentations:", len(test_idx))
    print("n_test trials:", len(test_trials))
    print("test trials:", test_trials)

sklearn: 1.7.2
X shape: (109, 333)

Fold 1
n_test presentations: 23
n_test trials: 16
test trials: [np.int64(15), np.int64(25), np.int64(26), np.int64(28), np.int64(30), np.int64(37), np.int64(38), np.int64(53), np.int64(58), np.int64(61), np.int64(66), np.int64(68), np.int64(82), np.int64(84), np.int64(91), np.int64(96)]

Fold 2
n_test presentations: 22
n_test trials: 17
test trials: [np.int64(5), np.int64(6), np.int64(7), np.int64(16), np.int64(27), np.int64(34), np.int64(36), np.int64(40), np.int64(43), np.int64(55), np.int64(59), np.int64(67), np.int64(76), np.int64(78), np.int64(79), np.int64(80), np.int64(97)]

Fold 3
n_test presentations: 21
n_test trials: 16
test trials: [np.int64(1), np.int64(13), np.int64(17), np.int64(24), np.int64(39), np.int64(48), np.int64(50), np.int64(51), np.int64(63), np.int64(72), np.int64(73), np.int64(77), np.int64(83), np.int64(85), np.int64(94), np.int64(98)]

Fold 4
n_test presentations: 22
n_test trials: 17
test trials: [np.int64(2), np.int64(8

In [26]:
from pathlib import Path
import pandas as pd
import numpy as np

RESULTS_DIR = PROJECT_ROOT / "tables1" / "population_decoding"

summary = pd.read_csv(
    RESULTS_DIR / "all_subjects_population_summary.csv"
)

summary

,subject,n_neurons,n_presentations,n_trials,n_features,lda_accuracy,lda_balanced_accuracy,logistic_accuracy,logistic_balanced_accuracy,nominal_chance
0,YFF,37,109,82,333,0.183550,0.138889,0.182684,0.142593,0.111111
1,YFI,29,104,76,261,0.183939,0.114815,0.173939,0.123704,0.111111
2,YFJ,45,114,78,405,0.148781,0.116898,0.112813,0.087500,0.111111
3,YFK,44,114,78,396,0.228228,0.197222,0.138933,0.117361,0.111111
4,YFL,56,114,78,504,0.158267,0.122917,0.122661,0.102315,0.111111
5,YFM,61,114,78,549,0.148748,0.119676,0.149506,0.130324,0.111111
6,YFP,43,158,109,387,0.171169,0.141481,0.132661,0.106667,0.111111
7,YFR,64,266,153,576,0.146471,0.134868,0.105241,0.097063,0.111111
8,YFS,59,264,156,531,0.091230,0.086667,0.064449,0.062963,0.111111
9,YFT,52,477,290,468,0.163443,0.159737,0.123618,0.122181,0.111111


In [27]:
metrics = [
    "lda_accuracy",
    "lda_balanced_accuracy",
    "logistic_accuracy",
    "logistic_balanced_accuracy",
]

overall = summary[metrics].agg(["mean", "std", "min", "max"]).T

overall["mean_percent"] = 100 * overall["mean"]
overall["std_percent"] = 100 * overall["std"]

overall.round(4)

,mean,std,min,max,mean_percent,std_percent
lda_accuracy,0.1553,0.0406,0.0850,0.2282,15.5345,4.0593
lda_balanced_accuracy,0.1288,0.0319,0.0840,0.1972,12.8829,3.1905
logistic_accuracy,0.1280,0.0336,0.0644,0.1827,12.7971,3.3555
logistic_balanced_accuracy,0.1084,0.0221,0.0630,0.1426,10.8437,2.2119


In [ ]:
comparison = summary[
    [
        "subject",
        "n_neurons",
        "n_presentations",
        "lda_accuracy",
        "lda_balanced_accuracy",
        "logistic_accuracy",
        "logistic_balanced_accuracy",
    ]
].copy()

for col in [
    "lda_accuracy",
    "lda_balanced_accuracy",
    "logistic_accuracy",
    "logistic_balanced_accuracy",
]:
    comparison[col] *= 100

comparison["LDA_minus_Logistic"] = (
    comparison["lda_accuracy"] - comparison["logistic_accuracy"]
)

comparison.round(2)

,subject,n_neurons,n_presentations,lda_accuracy,lda_balanced_accuracy,logistic_accuracy,logistic_balanced_accuracy,LDA_minus_Logistic
0,YFF,37,109,18.35,13.89,18.27,14.26,0.09
1,YFI,29,104,18.39,11.48,17.39,12.37,1.00
2,YFJ,45,114,14.88,11.69,11.28,8.75,3.60
3,YFK,44,114,22.82,19.72,13.89,11.74,8.93
4,YFL,56,114,15.83,12.29,12.27,10.23,3.56
5,YFM,61,114,14.87,11.97,14.95,13.03,-0.08
6,YFP,43,158,17.12,14.15,13.27,10.67,3.85
7,YFR,64,266,14.65,13.49,10.52,9.71,4.12
8,YFS,59,264,9.12,8.67,6.44,6.30,2.68
9,YFT,52,477,16.34,15.97,12.36,12.22,3.98


: 